# 📊 Notebook 3 — KPI Analysis
**Tracking, Benchmarking & Trending all Key Performance Indicators**

Sections:
1. Core KPI scorecard
2. Revenue KPIs (GMV, Net Revenue, AOV, Revenue per Customer)
3. Operational KPIs (Delivery Rate, Cancel Rate, Refund Rate)
4. Customer KPIs (Acquisition, Retention proxy, CLV)
5. Restaurant KPIs (Rating, Revenue per Brand, Coverage)
6. KPI heatmap by city & month
7. KPI trend lines with targets


In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (12, 5),
                     "axes.titlesize": 13, "axes.labelsize": 11})

BASE = r"C:\Users\rkuma\OneDrive\Desktop\Zomato"

def load(name):
    return pd.read_csv(os.path.join(BASE, f"Zomato  Order Data.xlsx - {name}.csv"))

customers   = load("Customer")
orders      = load("Orders")
restaurants = load("Restaurants")

orders["order_timestamp"] = pd.to_datetime(orders["order_timestamp"],
                                           format="%m/%d/%Y", errors="coerce")
orders["order_month"]   = orders["order_timestamp"].dt.to_period("M")
orders["order_quarter"] = orders["order_timestamp"].dt.to_period("Q")
orders["order_year"]    = orders["order_timestamp"].dt.year.astype("Int64")
orders["day_of_week"]   = orders["order_timestamp"].dt.day_name()
orders["hour"]          = orders["order_timestamp"].dt.hour
orders["discount_amount"] = orders["discount_amount"].fillna(0)
orders["delivery_fee"]    = orders["delivery_fee"].fillna(0)
orders["net_revenue"]     = orders["order_amount"] - orders["discount_amount"]
orders["is_discounted"]   = (orders["discount_amount"] > 0).astype(int)
orders["total_charge"]    = orders["net_revenue"] + orders["delivery_fee"]

customers["Signup_Time"]  = pd.to_datetime(customers["Signup_Time"],
                                           format="%d/%m/%Y", errors="coerce")
customers["signup_month"] = customers["Signup_Time"].dt.to_period("M")
customers["signup_year"]  = customers["Signup_Time"].dt.year.astype("Int64")

full = (orders
        .merge(restaurants, on="restaurant_id", how="left")
        .merge(customers,   left_on="customer_id",
               right_on="Customer_id", how="left"))

delivered  = full[full["order_status"] == "Delivered"].copy()
cancelled  = full[full["order_status"] == "Cancelled"].copy()
refunded   = full[full["order_status"] == "Refunded"].copy()

print(f"Orders: {len(orders):,} | Customers: {customers['Customer_id'].nunique():,} | Restaurants: {len(restaurants)}")
print(f"Date range: {orders['order_timestamp'].min().date()} to {orders['order_timestamp'].max().date()}")


## 1. Core KPI Scorecard

In [ ]:

total_orders    = len(orders)
total_customers = customers["Customer_id"].nunique()
gmv             = orders["order_amount"].sum()
net_revenue     = delivered["net_revenue"].sum()
avg_order_value = delivered["order_amount"].mean()
delivery_rate   = (orders["order_status"]=="Delivered").mean()*100
cancel_rate     = (orders["order_status"]=="Cancelled").mean()*100
refund_rate     = (orders["order_status"]=="Refunded").mean()*100
total_discounts = orders["discount_amount"].sum()
discount_rate   = (orders["discount_amount"]>0).mean()*100
avg_delivery_fee= orders["delivery_fee"].mean()
rev_per_cust    = net_revenue / total_customers
repeat_cust     = (delivered.groupby("customer_id")["order_id"].count() > 1).sum()
repeat_rate     = repeat_cust / total_customers * 100
avg_rating_all  = restaurants["avg_rating"].mean()

kpis = {
    "Total Orders":            (f"{total_orders:,}",         "🛒"),
    "Unique Customers":        (f"{total_customers:,}",      "👥"),
    "Gross Merchandise Value": (f"₹{gmv/1e6:.2f}M",         "💳"),
    "Net Revenue":             (f"₹{net_revenue/1e6:.2f}M",  "💰"),
    "Avg Order Value":         (f"₹{avg_order_value:,.0f}",  "🧾"),
    "Revenue per Customer":    (f"₹{rev_per_cust:,.0f}",     "📈"),
    "Delivery Rate":           (f"{delivery_rate:.1f}%",     "✅"),
    "Cancellation Rate":       (f"{cancel_rate:.1f}%",       "❌"),
    "Refund Rate":             (f"{refund_rate:.1f}%",       "↩️"),
    "Discount Usage Rate":     (f"{discount_rate:.1f}%",     "🏷️"),
    "Avg Delivery Fee":        (f"₹{avg_delivery_fee:.0f}",  "🛵"),
    "Repeat Customer Rate":    (f"{repeat_rate:.1f}%",       "🔄"),
    "Avg Restaurant Rating":   (f"{avg_rating_all:.2f}★",    "⭐"),
}

print("="*60)
print(f"{'KPI':<28} {'VALUE':>12}  {'ICON'}")
print("="*60)
for k,(v,icon) in kpis.items():
    print(f"  {k:<26} {v:>12}  {icon}")
print("="*60)


## 2. Revenue KPIs

In [ ]:

# GMV vs Net Revenue waterfall
gmv_val   = orders["order_amount"].sum()
disc_val  = orders["discount_amount"].sum()
net_val   = gmv_val - disc_val
rev_fee   = orders["delivery_fee"].sum()
lost_canc = cancelled["order_amount"].sum()
lost_ref  = refunded["order_amount"].sum()

labels  = ["GMV", "- Discounts", "Net Revenue", "+ Delivery Fees",
           "- Cancellations (lost)", "- Refunds (lost)", "Effective Revenue"]
values  = [gmv_val, -disc_val, net_val, rev_fee, -lost_canc, -lost_ref,
           net_val + rev_fee - lost_canc - lost_ref]
colors  = ["#2A9D8F","#E63946","#264653","#43AA8B",
           "#E63946","#9C6B98","#E9C46A"]

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(labels, [abs(v) for v in values], color=colors)
for bar, v in zip(bars, values):
    sign = "+" if v >= 0 else "-"
    ax.text(bar.get_x() + bar.get_width()/2,
            abs(v) + 200000,
            f"{sign}₹{abs(v)/1e6:.1f}M",
            ha="center", fontsize=9, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"₹{x/1e6:.0f}M"))
ax.set_title("Revenue Waterfall: GMV → Effective Revenue", fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


In [ ]:

# Monthly revenue KPIs
m_kpis = delivered.groupby("order_month").agg(
    gmv        = ("order_amount","sum"),
    net_rev    = ("net_revenue","sum"),
    orders_cnt = ("order_id","count"),
    aov        = ("order_amount","mean"),
    uniq_cust  = ("customer_id","nunique"),
).reset_index()
m_kpis["order_month"]  = m_kpis["order_month"].astype(str)
m_kpis["rev_per_cust"] = m_kpis["net_rev"] / m_kpis["uniq_cust"]
m_kpis["mom_rev_growth"] = m_kpis["net_rev"].pct_change()*100

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (col, label, color) in zip(axes.flat, [
    ("net_rev",      "Net Revenue (₹)",        "#264653"),
    ("aov",          "Avg Order Value (₹)",     "#2A9D8F"),
    ("uniq_cust",    "Active Customers",        "#E9C46A"),
    ("orders_cnt",   "Order Count",             "#43AA8B"),
    ("rev_per_cust", "Revenue per Customer (₹)","#E76F51"),
    ("mom_rev_growth","MoM Revenue Growth (%)", "#457B9D"),
]):
    ax.plot(range(len(m_kpis)), m_kpis[col].fillna(0),
            marker="o", color=color, linewidth=2)
    ax.set_xticks(range(len(m_kpis)))
    ax.set_xticklabels(m_kpis["order_month"], rotation=60, fontsize=7)
    ax.set_title(label, fontweight="bold")
    if "rev" in col.lower() and "growth" not in col.lower():
        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(lambda x,_: f"₹{x/1e6:.1f}M" if x >= 1e6 else f"₹{x:,.0f}"))

plt.suptitle("Monthly Revenue KPI Trends", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 3. Operational KPIs

In [ ]:

# Monthly delivery, cancel, refund rates
m_ops = orders.groupby("order_month").agg(
    total      = ("order_id","count"),
    delivered  = ("order_status", lambda x: (x=="Delivered").sum()),
    cancelled  = ("order_status", lambda x: (x=="Cancelled").sum()),
    refunded   = ("order_status", lambda x: (x=="Refunded").sum()),
).reset_index()
m_ops["order_month"]    = m_ops["order_month"].astype(str)
m_ops["delivery_rate"]  = m_ops["delivered"]  / m_ops["total"] * 100
m_ops["cancel_rate"]    = m_ops["cancelled"]  / m_ops["total"] * 100
m_ops["refund_rate"]    = m_ops["refunded"]   / m_ops["total"] * 100

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(m_ops))
ax.plot(x, m_ops["delivery_rate"], marker="o", label="Delivery Rate %",
        color="#43AA8B", linewidth=2)
ax.plot(x, m_ops["cancel_rate"],   marker="s", label="Cancel Rate %",
        color="#E63946", linewidth=2)
ax.plot(x, m_ops["refund_rate"],   marker="^", label="Refund Rate %",
        color="#9C6B98", linewidth=2)
ax.axhline(60, color="#43AA8B", linestyle="--", linewidth=1, alpha=0.5, label="Delivery Target 60%")
ax.axhline(15, color="#E63946", linestyle="--", linewidth=1, alpha=0.5, label="Cancel Target 15%")
ax.axhline(10, color="#9C6B98", linestyle="--", linewidth=1, alpha=0.5, label="Refund Target 10%")
ax.set_xticks(x)
ax.set_xticklabels(m_ops["order_month"], rotation=60, fontsize=8)
ax.set_title("Operational KPI Trends vs Targets", fontsize=13, fontweight="bold")
ax.set_ylabel("%")
ax.legend(ncol=3, fontsize=9)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()


In [ ]:

# City-level operational KPI table
city_ops = full.groupby("City").agg(
    total_orders   = ("order_id","count"),
    delivery_rate  = ("order_status", lambda x: (x=="Delivered").sum()/len(x)*100),
    cancel_rate    = ("order_status", lambda x: (x=="Cancelled").sum()/len(x)*100),
    refund_rate    = ("order_status", lambda x: (x=="Refunded").sum()/len(x)*100),
    avg_order_val  = ("order_amount","mean"),
    avg_rating     = ("avg_rating","mean"),
).reset_index().round(2).sort_values("delivery_rate", ascending=False)

display(city_ops.set_index("City"))

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(city_ops))
w = 0.25
ax.bar(x - w, city_ops["delivery_rate"],  w, label="Delivery %",  color="#43AA8B")
ax.bar(x,     city_ops["cancel_rate"],    w, label="Cancel %",    color="#E63946")
ax.bar(x + w, city_ops["refund_rate"],    w, label="Refund %",    color="#9C6B98")
ax.set_xticks(x)
ax.set_xticklabels(city_ops["City"], rotation=20)
ax.set_title("City-level Operational KPIs", fontsize=13, fontweight="bold")
ax.set_ylabel("%"); ax.legend()
plt.tight_layout()
plt.show()


## 4. Customer KPIs

In [ ]:

# Monthly signups & cumulative customers
m_signup = customers.groupby("signup_month")["Customer_id"].count().reset_index()
m_signup.columns = ["month","new_customers"]
m_signup["cumulative"] = m_signup["new_customers"].cumsum()
m_signup["month"] = m_signup["month"].astype(str)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.bar(m_signup["month"], m_signup["new_customers"],
        color="#264653", alpha=0.7, label="New Customers")
ax2.plot(m_signup["month"], m_signup["cumulative"],
         marker="o", color="#E63946", linewidth=2, label="Cumulative")
ax1.set_xticklabels(m_signup["month"], rotation=60, fontsize=8)
ax1.set_title("Monthly New Signups & Cumulative Growth", fontsize=13, fontweight="bold")
ax1.set_ylabel("New Customers"); ax2.set_ylabel("Cumulative Customers")
ax1.legend(loc="upper left"); ax2.legend(loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:

# CLV segments (RFM-lite)
clv = delivered.groupby("customer_id").agg(
    frequency = ("order_id","count"),
    monetary  = ("net_revenue","sum"),
    recency   = ("order_timestamp","max"),
).reset_index()
clv["recency_days"] = (delivered["order_timestamp"].max() - clv["recency"]).dt.days

# Segment by frequency
clv["segment"] = pd.cut(clv["frequency"],
                         bins=[0,1,3,7,999],
                         labels=["One-time","Occasional","Regular","Champion"])

seg_stats = clv.groupby("segment", observed=True).agg(
    customers = ("customer_id","count"),
    avg_orders= ("frequency","mean"),
    avg_rev   = ("monetary","mean"),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].bar(seg_stats["segment"].astype(str), seg_stats["customers"],
            color=sns.color_palette("Set2",4))
axes[0].set_title("Customers per Segment")
for i, v in enumerate(seg_stats["customers"]):
    axes[0].text(i, v+2, str(v), ha="center", fontweight="bold")

axes[1].bar(seg_stats["segment"].astype(str), seg_stats["avg_orders"],
            color=sns.color_palette("Pastel1",4))
axes[1].set_title("Avg Orders per Segment")

axes[2].bar(seg_stats["segment"].astype(str), seg_stats["avg_rev"],
            color=sns.color_palette("YlOrRd",4))
axes[2].set_title("Avg Revenue per Segment (₹)")
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"₹{x:,.0f}"))

plt.suptitle("Customer Segmentation KPIs", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print(seg_stats.to_string(index=False))


## 5. Restaurant KPIs

In [ ]:

rest_kpis = full.groupby("restaurant_name").agg(
    total_orders  = ("order_id","count"),
    delivered     = ("order_status", lambda x: (x=="Delivered").sum()),
    net_revenue   = ("net_revenue","sum"),
    avg_rating    = ("avg_rating","mean"),
    cities_served = ("City","nunique"),
    refund_rate   = ("order_status", lambda x: (x=="Refunded").sum()/len(x)*100),
).reset_index()
rest_kpis["delivery_rate"] = rest_kpis["delivered"]/rest_kpis["total_orders"]*100
rest_kpis["rev_per_order"] = rest_kpis["net_revenue"]/rest_kpis["delivered"]
rest_kpis = rest_kpis.sort_values("net_revenue", ascending=False)

display(rest_kpis.set_index("restaurant_name").round(2))


In [ ]:

# KPI spider/radar chart for top 5 brands
from matplotlib.patches import FancyArrowPatch

top5 = rest_kpis.head(5)["restaurant_name"].tolist()
top5_df = rest_kpis[rest_kpis["restaurant_name"].isin(top5)].copy()
metrics_r = ["delivery_rate","avg_rating","cities_served","total_orders"]

# normalise
for m in metrics_r:
    col_max = top5_df[m].max()
    top5_df[m+"_norm"] = top5_df[m] / col_max * 100

categories = [m.replace("_"," ").title() for m in metrics_r]
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors_r = sns.color_palette("Set2", 5)
for i, (_, row) in enumerate(top5_df.iterrows()):
    vals = [row[m+"_norm"] for m in metrics_r]
    vals += vals[:1]
    ax.plot(angles, vals, "o-", linewidth=2, color=colors_r[i],
            label=row["restaurant_name"])
    ax.fill(angles, vals, alpha=0.1, color=colors_r[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_title("Top 5 Brands — KPI Radar Chart
(normalised 0–100)", fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()


## 6. KPI Heatmap — City × Month

In [ ]:

def city_month_kpi(metric_fn, title):
    pivot = (full.groupby(["City","order_month"])
             .apply(metric_fn)
             .unstack())
    pivot.columns = pivot.columns.astype(str)
    fig, ax = plt.subplots(figsize=(18, 6))
    sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd",
                linewidths=0.4, ax=ax, cbar_kws={"label": title})
    ax.set_title(f"City × Month — {title}", fontweight="bold", fontsize=13)
    ax.set_xlabel("Month"); ax.set_ylabel("City")
    plt.xticks(rotation=60, fontsize=7)
    plt.tight_layout()
    plt.show()

city_month_kpi(lambda g: (g["order_status"]=="Delivered").sum()/len(g)*100,
               "Delivery Rate (%)")
city_month_kpi(lambda g: (g["order_status"]=="Cancelled").sum()/len(g)*100,
               "Cancel Rate (%)")


## ✅ KPI Analysis Summary
- **Delivery rate ~59.7%** is below a healthy 70%+ target — operational gaps exist.
- **Cancel + Refund combined = 40.3%** of orders are non-fulfilling, representing significant revenue leakage.
- **AOV is stable ~₹900** with minimal variation across cities — pricing is consistent.
- **Revenue per customer** trends upward with tenure, validating investment in retention.
- **Champion customers** (>7 orders) are few but generate 3–5x the revenue of one-timers.
- **City × Month heatmaps** reveal specific city-month combinations with sustained high cancellation rates.